In [1]:
from google.colab import files
uploaded = files.upload()

Saving dataset_amb.mat to dataset_amb.mat
Saving dataset_elec.mat to dataset_elec.mat


In [2]:
import scipy.io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Models to compare
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# --- 1. Load & Preprocess (Your existing logic) ---
elec = scipy.io.loadmat('dataset_elec.mat')
amb = scipy.io.loadmat('dataset_amb.mat')

df = pd.DataFrame({
    'vdc1': elec['vdc1'].flatten(), 'vdc2': elec['vdc2'].flatten(),
    'idc1': elec['idc1'].flatten(), 'idc2': elec['idc2'].flatten(),
    'irr': amb['irr'].flatten(), 'pvt': amb['pvt'].flatten(),
    'label': amb['f_nv'].flatten()
})

def assign_severity(row):
    lbl = row['label']
    if lbl == 0: return 0
    if lbl == 1:
        v_diff = abs(row['vdc1'] - row['vdc2'])
        return 3 if v_diff > 60 else (2 if v_diff > 40 else 1)
    if lbl == 3:
        irr = row['irr']
        return 3 if irr >= 800 else (2 if irr >= 400 else 1)
    return -1

df['severity_target'] = df.apply(assign_severity, axis=1)
df_ml = df[df['severity_target'] != -1].copy()
df_final = pd.concat([
    df_ml[df_ml['severity_target'] == 0].sample(n=20000, random_state=42),
    df_ml[df_ml['severity_target'] > 0]
])

X = df_final[['vdc1', 'vdc2', 'idc1', 'idc2', 'irr', 'pvt']]
y = df_final['severity_target']

# --- 2. Scaling (Essential for SVM and KNN) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

# --- 3. Model Training and Evaluation Loop ---
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', C=1.0, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Collect metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro'),
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro'),
        "F1-Score (Macro)": f1_score(y_test, y_pred, average='macro')
    })

# --- 4. Comparison Table ---
comparison_df = pd.DataFrame(results).sort_values(by="F1-Score (Macro)", ascending=False)
print("\n--- Model Evaluation Comparison ---")
print(comparison_df.to_string(index=False))

Training Random Forest...
Training Gradient Boosting...
Training SVM (RBF)...
Training KNN (k=5)...

--- Model Evaluation Comparison ---
            Model  Accuracy  Precision (Macro)  Recall (Macro)  F1-Score (Macro)
    Random Forest  0.993859           0.974027        0.973726          0.973876
        KNN (k=5)  0.987821           0.962615        0.948561          0.955313
Gradient Boosting  0.992714           0.905644        0.874223          0.888068
        SVM (RBF)  0.943166           0.914152        0.823127          0.842940
